In [26]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import scipy.stats as stats

# ==========================================
# 1. CONFIGURATION & SEED
# ==========================================
class Config:
    SEED = 42
    NUM_FACILITIES = 100 # Start small for testing, scale to 5000+ later
    START_DATE = pd.to_datetime('2023-01-01')
    END_DATE = pd.to_datetime('2023-12-31')
    
    # Medicine definitions
    MEDICINES = ['PARACETAMOL', 'RIF_INH', 'EPINEPHRINE']
    
    @classmethod
    def get_rng(cls):
        """Returns a reproducible numpy random generator"""
        return np.random.default_rng(cls.SEED)

# ==========================================
# 2. GENERATE FACILITIES (Static Data)
# ==========================================
def generate_facilities(config):
    rng = config.get_rng()
    n = config.NUM_FACILITIES
    
    # Generate Base IDs
    facility_ids = [f"FAC-{str(i).zfill(4)}" for i in range(1, n + 1)]
    
    # 60% Rural, 40% Urban (Bernoulli/Binomial distribution)
    is_rural = rng.binomial(1, 0.6, size=n)
    region_type = np.where(is_rural == 1, 'Rural', 'Urban')
    facility_type = np.where(is_rural == 1, 'BHS', 'RHU')
    
    # Latent Management Quality (Beta distribution: heavily skewed towards good, but with a left tail)
    # 0 = terrible, 1 = perfect
    latent_quality = rng.beta(a=5, b=2, size=n)
    
    # Population Served (Truncated Normal Distribution based on Facility Type)
    # BHS: mean 5000, RHU: mean 22000
    bhs_pop = rng.normal(loc=5000, scale=800, size=n)
    rhu_pop = rng.normal(loc=22000, scale=4000, size=n)
    
    population = np.where(facility_type == 'BHS', bhs_pop, rhu_pop)
    population = np.clip(population, 2500, 35000).astype(int) # Ensure realistic bounds
    
    # Storage Capacity (Highly correlated with population and facility type)
    storage = (population * 0.5 * rng.uniform(0.8, 1.2, size=n)).astype(int)
    
    # Assemble Dataframe
    df_facilities = pd.DataFrame({
        'facility_id': facility_ids,
        'facility_type': facility_type,
        'region_type': region_type,
        'population_served': population,
        'storage_capacity_liters': storage,
        'reorder_interval_days': 30, # Standard push cycle
        'latent_management_quality': latent_quality # OMITTED VARIABLE (Keep for now)
    })
    
    return df_facilities

# Run and inspect
df_facilities = generate_facilities(Config)
display(df_facilities.head())
# display(df_facilities['latent_management_quality'].hist()) # Optional: visualize the distribution!

,facility_id,facility_type,region_type,population_served,storage_capacity_liters,reorder_interval_days,latent_management_quality
0,FAC-0001,RHU,Urban,17110,8598,30,0.820159
1,FAC-0002,BHS,Rural,5293,2575,30,0.843894
2,FAC-0003,RHU,Urban,29285,11923,30,0.491247
3,FAC-0004,RHU,Urban,15392,9111,30,0.848628
4,FAC-0005,BHS,Rural,4753,1999,30,0.708436


In [27]:
import itertools

# ==========================================
# 3. GENERATE OUTBREAKS & CONSUMPTION
# ==========================================
def generate_consumption(config, df_facilities):
    rng = config.get_rng()
    
    
    # 1. Create the Time-Series Grid (Cartesian Product)
    date_range = pd.date_range(start=config.START_DATE, end=config.END_DATE)
    
    # Create all combinations of (Facility, Date)
    grid = list(itertools.product(df_facilities['facility_id'], date_range))
    df_grid = pd.DataFrame(grid, columns=['facility_id', 'date'])
    
    # Merge in facility details we need for the math
    df_grid = df_grid.merge(
            df_facilities[['facility_id', 'region_type', 'population_served', 'latent_management_quality']], 
            on='facility_id', 
            how='left'
        )
    
    # 2. Generate the Latent Outbreak Intensity (Omitted Variable)
    # We model a sine wave peaking around day 240 (August/September - rainy season)
    df_grid['day_of_year'] = df_grid['date'].dt.dayofyear
    
    # Random amplitude for Rural vs Urban
    A_rural = rng.uniform(0.6, 0.9)
    A_urban = rng.uniform(0.4, 0.7)
    df_grid['amplitude'] = np.where(df_grid['region_type'] == 'Rural', A_rural, A_urban)
    
    # The seasonal curve + random daily noise
    seasonality = np.sin(2 * np.pi * (df_grid['day_of_year'] - 150) / 365)
    noise = rng.normal(0, 0.05, size=len(df_grid))
    
    df_grid['latent_outbreak_intensity'] = np.clip((seasonality * df_grid['amplitude']) + noise, 0, 1)
    
    # 3. Daily Patient Visits
    lambda_visits = df_grid['population_served'] * (0.003 + 0.005 * df_grid['latent_outbreak_intensity'])
    df_grid['patient_consultations'] = rng.poisson(lambda_visits)
    
    # 4. Paracetamol (Spiky, outbreak-driven)
    lambda_para = df_grid['patient_consultations'] * (1.8 + 3.2 * df_grid['latent_outbreak_intensity'])
    df_grid['PARACETAMOL_dispensed'] = rng.poisson(lambda_para)
    
    # 5. TB Meds (Steady, but adherence drops if facility management is poor)
    # Adherence is between 50% (poorly managed) and 100% (perfectly managed)
    adherence = 0.5 + (0.5 * df_grid['latent_management_quality'])
    lambda_tb = df_grid['population_served'] * 0.0004 * adherence
    df_grid['RIF_INH_dispensed'] = rng.poisson(lambda_tb)
    
    # 6. Epinephrine (Rare, acute emergency)
    lambda_epi = np.maximum(0.02, df_grid['population_served'] * 0.00002)
    df_grid['EPINEPHRINE_dispensed'] = rng.poisson(lambda_epi)
    
    # 7. Reshape (Melt) to Long Format
    value_vars = ['PARACETAMOL_dispensed', 'RIF_INH_dispensed', 'EPINEPHRINE_dispensed']
    df_consumption = pd.melt(
        df_grid, 
        id_vars=['facility_id', 'date', 'patient_consultations', 'latent_outbreak_intensity'],
        value_vars=value_vars,
        var_name='medicine_code', 
        value_name='quantity_dispensed'
    )
 
    df_consumption['medicine_code'] = df_consumption['medicine_code'].str.replace('_dispensed', '')
    
    df_consumption['consumption_id'] = [f"CON-{str(i).zfill(7)}" for i in range(1, len(df_consumption) + 1)]
    
    return df_consumption

df_consumption = generate_consumption(Config, df_facilities)
print(df_consumption.head())

  facility_id       date  patient_consultations  latent_outbreak_intensity  \
0    FAC-0001 2023-01-01                     48                        0.0   
1    FAC-0001 2023-01-02                     62                        0.0   
2    FAC-0001 2023-01-03                     57                        0.0   
3    FAC-0001 2023-01-04                     53                        0.0   
4    FAC-0001 2023-01-05                     61                        0.0   

  medicine_code  quantity_dispensed consumption_id  
0   PARACETAMOL                 108    CON-0000001  
1   PARACETAMOL                 123    CON-0000002  
2   PARACETAMOL                 113    CON-0000003  
3   PARACETAMOL                  88    CON-0000004  
4   PARACETAMOL                 113    CON-0000005  


In [28]:
# ==========================================
# 4. GENERATE DELIVERIES
# ==========================================
def generate_deliveries(config, df_facilities):
    rng = config.get_rng()
    
    # Create expected delivery dates (every 30 days)
    delivery_dates = pd.date_range(start=config.START_DATE, end=config.END_DATE, freq='30D')
    
    # Cross join Facilities x Medicines x Delivery Dates
    grid = list(itertools.product(df_facilities['facility_id'], config.MEDICINES, delivery_dates))
    df_del = pd.DataFrame(grid, columns=['facility_id', 'medicine_code', 'expected_delivery_date'])
    
    # Merge facility attributes
    df_del = df_del.merge(
        df_facilities[['facility_id', 'region_type', 'latent_management_quality']], 
        on='facility_id', how='left'
    )
    
    # Calculate Mean Delay (mu) based on Word Doc formula
    # mu = 3.0 + 8.0(Rural) + 14.0(1 - Quality)
    is_rural = (df_del['region_type'] == 'Rural').astype(int)
    mu_delay = 3.0 + (8.0 * is_rural) + (14.0 * (1 - df_del['latent_management_quality']))
    
    # Apply Exponential Distribution for actual delay days (needs scale = mu)
    delay_days = rng.exponential(scale=mu_delay).astype(int)
    
    df_del['actual_delivery_date'] = df_del['expected_delivery_date'] + pd.to_timedelta(delay_days, unit='D')
    df_del['transit_delay_days'] = delay_days
    
    # Standard dispatch quantities (simplification: base it on medicine type)
    base_qty = {'PARACETAMOL': 5000, 'RIF_INH': 800, 'EPINEPHRINE': 10}
    df_del['quantity_dispatched'] = df_del['medicine_code'].map(base_qty)
    
    # Add primary key
    df_del['delivery_id'] = [f"DEL-{str(i).zfill(6)}" for i in range(1, len(df_del) + 1)]
    
    # Status labeling
    df_del['delivery_status'] = np.where(df_del['transit_delay_days'] <= 3, 'On-Time', 'Delayed')
    
    # Clean up (drop temporary columns used for math)
    df_del = df_del.drop(columns=['region_type', 'latent_management_quality'])
    
    return df_del

# RUN IT
df_deliveries = generate_deliveries(Config, df_facilities)
display(df_deliveries.head())

,facility_id,medicine_code,expected_delivery_date,actual_delivery_date,transit_delay_days,quantity_dispatched,delivery_id,delivery_status
0,FAC-0001,PARACETAMOL,2023-01-01,2023-01-14,13,5000,DEL-000001,Delayed
1,FAC-0001,PARACETAMOL,2023-01-31,2023-02-12,12,5000,DEL-000002,Delayed
2,FAC-0001,PARACETAMOL,2023-03-02,2023-03-15,13,5000,DEL-000003,Delayed
3,FAC-0001,PARACETAMOL,2023-04-01,2023-04-02,1,5000,DEL-000004,On-Time
4,FAC-0001,PARACETAMOL,2023-05-01,2023-05-01,0,5000,DEL-000005,On-Time


In [29]:
# ==========================================
# 5. GENERATE INVENTORY & AUDITS
# ==========================================
def generate_inventory_and_stockouts(config, df_facilities, df_consumption, df_deliveries):
    rng = config.get_rng()
    
    # 1. Prepare Daily Inflows and Outflows
    # Deliveries (Inflow)
    df_del_daily = df_deliveries[['facility_id', 'medicine_code', 'actual_delivery_date', 'quantity_dispatched']].copy()
    df_del_daily.rename(columns={'actual_delivery_date': 'date', 'quantity_dispatched': 'delivered_qty'}, inplace=True)
    
    # FIX: Group by to sum up deliveries if two shipments randomly arrive on the same day!
    df_del_daily = df_del_daily.groupby(['facility_id', 'medicine_code', 'date'], as_index=False)['delivered_qty'].sum()
    
    # Consumption (Outflow)
    df_cons_daily = df_consumption[['facility_id', 'medicine_code', 'date', 'quantity_dispensed']].copy()
    
    # FIX: Group by to ensure strict uniqueness
    df_cons_daily = df_cons_daily.groupby(['facility_id', 'medicine_code', 'date'], as_index=False)['quantity_dispensed'].sum()
    
    # Merge into a single daily timeline grid
    date_range = pd.date_range(start=config.START_DATE, end=config.END_DATE)
    grid = list(itertools.product(df_facilities['facility_id'], config.MEDICINES, date_range))
    df_ledger = pd.DataFrame(grid, columns=['facility_id', 'medicine_code', 'date'])
    
    # Attach Latent Quality for our phantom stock math
    df_ledger = df_ledger.merge(df_facilities[['facility_id', 'latent_management_quality']], on='facility_id', how='left')
    
    # Merge Inflows and Outflows
    df_ledger = df_ledger.merge(df_del_daily, on=['facility_id', 'medicine_code', 'date'], how='left').fillna({'delivered_qty': 0})
    df_ledger = df_ledger.merge(df_cons_daily, on=['facility_id', 'medicine_code', 'date'], how='left').fillna({'quantity_dispensed': 0})
    
    # Sort chronologically to prepare for the Time-Step Loop
    df_ledger = df_ledger.sort_values(['facility_id', 'medicine_code', 'date']).reset_index(drop=True)
    
    # 2. The Time-Step Loop (Simulate the physical shelves day by day)
    actual_stock = np.zeros(len(df_ledger)) # Array to hold our running balance
    unmet_demand = np.zeros(len(df_ledger)) # Array to hold stockout shortages
    
    # Group indices by date to update all facilities simultaneously
    date_indices = df_ledger.groupby('date').indices
    dates_sorted = sorted(date_indices.keys())
    
    current_stock = np.zeros(len(df_facilities) * len(config.MEDICINES)) # Starting stock (assume 0 for simplicity, or add a starting buffer)
    
    for dt in dates_sorted:
        idx = date_indices[dt]
        
        deliveries_today = df_ledger.loc[idx, 'delivered_qty'].values
        demand_today = df_ledger.loc[idx, 'quantity_dispensed'].values
        quality = df_ledger.loc[idx, 'latent_management_quality'].values
        
        # Add deliveries
        current_stock += deliveries_today
        
        # Shrinkage (Spoilage/Loss driven by poor management)
        # Using binomial: if you have 100 boxes, what is the chance each box spoils?
        prob_spoil = 0.002 * (1 - quality)
        shrinkage = rng.binomial(current_stock.astype(int), prob_spoil)
        current_stock -= shrinkage
        
        # Subtract consumption (can't go below 0)
        stock_after_demand = current_stock - demand_today
        unmet = np.where(stock_after_demand < 0, np.abs(stock_after_demand), 0)
        current_stock = np.maximum(0, stock_after_demand)
        
        # Store in our main arrays
        actual_stock[idx] = current_stock
        unmet_demand[idx] = unmet

    df_ledger['latent_actual_physical_stock'] = actual_stock
    df_ledger['unmet_demand'] = unmet_demand
    
    # 3. Generate End-of-Month Audits (inventory_levels table)
    # Filter only to end of month dates
    df_audits = df_ledger[df_ledger['date'].dt.is_month_end].copy()
    
    # Phantom Stock / Discrepancy (Math from Word Doc: high error for poor quality)
    quality_audit = df_audits['latent_management_quality']
    mu_error = 15 * (1 - quality_audit)
    sigma_error = 8 * (1 - quality_audit) + 1
    
    phantom_error = rng.normal(loc=mu_error, scale=sigma_error).astype(int)
    
    df_audits['latent_phantom_stock'] = phantom_error
    df_audits['recorded_stock_balance'] = np.maximum(0, df_audits['latent_actual_physical_stock'] + phantom_error)
    
    df_audits['audit_id'] = [f"AUD-{str(i).zfill(6)}" for i in range(1, len(df_audits) + 1)]
    df_audits.rename(columns={'date': 'audit_date'}, inplace=True)
    
    # Keep only the columns needed for the final schema + omitted variables
    df_inventory = df_audits[['audit_id', 'facility_id', 'medicine_code', 'audit_date', 
                              'recorded_stock_balance', 'latent_actual_physical_stock', 'latent_phantom_stock']]
    
    return df_ledger, df_inventory

# RUN IT
df_ledger, df_inventory = generate_inventory_and_stockouts(Config, df_facilities, df_consumption, df_deliveries)
display(df_inventory.head())

,audit_id,facility_id,medicine_code,audit_date,recorded_stock_balance,latent_actual_physical_stock,latent_phantom_stock
30,AUD-000001,FAC-0001,EPINEPHRINE,2023-01-31,12.0,11.0,1
58,AUD-000002,FAC-0001,EPINEPHRINE,2023-02-28,5.0,5.0,0
89,AUD-000003,FAC-0001,EPINEPHRINE,2023-03-31,8.0,8.0,0
119,AUD-000004,FAC-0001,EPINEPHRINE,2023-04-30,9.0,7.0,2
150,AUD-000005,FAC-0001,EPINEPHRINE,2023-05-31,6.0,0.0,6


In [30]:
# ==========================================
# 6. EXTRACT STOCKOUT EVENTS
# ==========================================
def extract_stockouts(df_ledger):
    # Filter to days where demand could not be met because stock was 0
    df_stockouts = df_ledger[df_ledger['unmet_demand'] > 0].copy()
    
    # For a junior level, we'll keep it simple: log every distinct day of stockout as an event.
    # (A more complex approach groups contiguous days, but daily logging is highly realistic for public health alert systems).
    
    df_stockouts['event_id'] = [f"STK-{str(i).zfill(6)}" for i in range(1, len(df_stockouts) + 1)]
    df_stockouts.rename(columns={'date': 'stockout_date', 'unmet_demand': 'estimated_unmet_demand'}, inplace=True)
    
    df_stockouts = df_stockouts[['event_id', 'facility_id', 'medicine_code', 'stockout_date', 'estimated_unmet_demand']]
    
    return df_stockouts

# RUN IT
df_stockouts = extract_stockouts(df_ledger)
display(df_stockouts.head())

,event_id,facility_id,medicine_code,stockout_date,estimated_unmet_demand
1,STK-000001,FAC-0001,EPINEPHRINE,2023-01-02,2.0
365,STK-000002,FAC-0001,PARACETAMOL,2023-01-01,108.0
366,STK-000003,FAC-0001,PARACETAMOL,2023-01-02,123.0
367,STK-000004,FAC-0001,PARACETAMOL,2023-01-03,113.0
368,STK-000005,FAC-0001,PARACETAMOL,2023-01-04,88.0


In [32]:
# ==========================================
# 7. CLEANUP & EXPORT
# ==========================================
import os

# Put all our dataframes into a dictionary
final_dfs = {
    'facilities': df_facilities,
    'deliveries': df_deliveries,
    'consumption': df_consumption,
    'inventory_levels': df_inventory,
    'stockout_events': df_stockouts
}

# Create the output folder if it does not exist.
# to_csv uses write mode by default, so existing files are overwritten.
output_dir = 'SyntheticDataset'
os.makedirs(output_dir, exist_ok=True)

print(f"Exporting datasets to {output_dir}/...")
for name, df in final_dfs.items():
    # 1. Identify any columns containing 'latent_'
    cols_to_drop = [c for c in df.columns if 'latent_' in c]
    
    # 2. Drop those columns
    df_clean = df.drop(columns=cols_to_drop)
    
    # 3. Save to CSV, overwriting an existing file with the same name
    filename = f"{name}.csv"
    output_path = os.path.join(output_dir, filename)
    df_clean.to_csv(output_path, index=False)
    
    print(f"  -> Saved {output_path} | Rows: {len(df_clean)} | Dropped {len(cols_to_drop)} latent columns")

print("\nNotebook 1 Complete! You have successfully generated the synthetic database.")

Exporting datasets to SyntheticDataset/...
  -> Saved SyntheticDataset\facilities.csv | Rows: 100 | Dropped 1 latent columns
  -> Saved SyntheticDataset\deliveries.csv | Rows: 3900 | Dropped 0 latent columns
  -> Saved SyntheticDataset\consumption.csv | Rows: 109500 | Dropped 1 latent columns
  -> Saved SyntheticDataset\inventory_levels.csv | Rows: 3600 | Dropped 2 latent columns
  -> Saved SyntheticDataset\stockout_events.csv | Rows: 5798 | Dropped 0 latent columns

Notebook 1 Complete! You have successfully generated the synthetic database.
